# Model training and comparison

### Packages Import

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split

from ml_powerlifting_potential.config import FEATURES, ROOT, SEED, TARGETS
from ml_powerlifting_potential.data import load_dataset
from ml_powerlifting_potential.models import evaluate, make_poly_pipeline, make_svr

### Data Import & Train/Test Split

In [ ]:
dataset = load_dataset()

X = dataset[FEATURES]
y_squat = dataset[TARGETS[0]]
y_deadlift = dataset[TARGETS[1]]
X_train, X_test, y_squat_train, y_squat_test, y_deadlift_train, y_deadlift_test = (
    train_test_split(X, y_squat, y_deadlift, test_size=0.2, random_state=SEED)
)

bench_train = X_train["Best3BenchKg"]
bench_test = X_test["Best3BenchKg"]

print(f"train: {len(X_train)} lifters, test: {len(X_test)} lifters")

### Ratio Estimator Method

In [ ]:
# Gym rule: median of the per-athlete ratios, computed on the TRAIN set only
ratio_squat = (y_squat_train / bench_train).median()
ratio_deadlift = (y_deadlift_train / bench_train).median()
print(f"squat/bench = {ratio_squat:.2f}, deadlift/bench = {ratio_deadlift:.2f}")

# Predictions on the TEST set
pred_squat_rule = bench_test * ratio_squat
pred_deadlift_rule = bench_test * ratio_deadlift

print("Squat rule:", evaluate(y_squat_test, pred_squat_rule))
print("Deadlift rule:", evaluate(y_deadlift_test, pred_deadlift_rule))

### Linear Regression Model

In [ ]:
linear_model_squat = LinearRegression()
linear_model_squat.fit(X_train, y_squat_train)

linear_model_deadlift = LinearRegression()
linear_model_deadlift.fit(X_train, y_deadlift_train)

pred_squat_linear = linear_model_squat.predict(X_test)
pred_deadlift_linear = linear_model_deadlift.predict(X_test)

print("Squat linear:", evaluate(y_squat_test, pred_squat_linear))
print("Deadlift linear:", evaluate(y_deadlift_test, pred_deadlift_linear))

### Polynomial Regression Model

In [ ]:
targets = {
    "squat": (y_squat_train, y_squat_test),
    "deadlift": (y_deadlift_train, y_deadlift_test),
}

# train and test scores, to check for overfitting
results = []
for degree in [2, 3]:
    for target_name, (y_train, y_test) in targets.items():
        model = make_poly_pipeline(degree)
        model.fit(X_train, y_train)
        for split_name, X_part, y_part in [
            ("train", X_train, y_train),
            ("test", X_test, y_test),
        ]:
            scores = evaluate(y_part, model.predict(X_part))
            results.append(
                {
                    "model": f"poly{degree}",
                    "target": target_name,
                    "split": split_name,
                    **scores,
                }
            )

results_df = pd.DataFrame(results)
results_df

### Finding the best SVR params using GridSearch for the Squat Prediction

In [ ]:
# values to try for each setting
param_grid = {
    "regressor__svr__C": [1, 10, 100],
    "regressor__svr__gamma": [0.05, 0.2, 0.5],
    "regressor__svr__epsilon": [0.05, 0.1],
}

# try every combination with 3-fold cross validation
grid_search_squat = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",  # negative because sklearn looks for the highest score
    n_jobs=-1,  # use all cpu cores
)

# tune on 8000 lifters only, the full train set would take too long
grid_search_squat.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])

print("Best SVR parameters for squat:")
print(grid_search_squat.best_params_)
print("Best MAE (kg):")
print(-grid_search_squat.best_score_)  # best MAE in kg

In [ ]:
# second pass, grid moved toward smaller gamma and bigger epsilon
param_grid = {
    "regressor__svr__C": [5, 10, 30],
    "regressor__svr__gamma": [0.01, 0.02, 0.05],
    "regressor__svr__epsilon": [0.1, 0.2],
}

grid_search_squat = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search_squat.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])

# read each param by its name
best_params = grid_search_squat.best_params_
best_SVR_C = best_params["regressor__svr__C"]
best_SVR_gamma = best_params["regressor__svr__gamma"]
best_SVR_epsilon = best_params["regressor__svr__epsilon"]

print("Best SVR parameters for squat:")
print(best_params)
print("Best MAE (kg):")
print(-grid_search_squat.best_score_)

### Finding the best SVR params using GridSearch for the Deadlift prediction

In [ ]:
# small grid around the squat best params
param_grid = {
    "regressor__svr__C": [10, 30, 100],
    "regressor__svr__gamma": [0.02, 0.05],
    "regressor__svr__epsilon": [0.1],
}

grid_search_deadlift = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search_deadlift.fit(X_train.iloc[:8_000], y_deadlift_train.iloc[:8_000])

print("Best SVR parameters for deadlift:")
print(grid_search_deadlift.best_params_)
print("Best MAE (kg):")
print(-grid_search_deadlift.best_score_)

# the best params for squat and deadlift are identical, so we use the same SVR settings for both lifts

### Best SVR on the full test set

In [ ]:
# best svr from the grid search, trained on 20000 lifters
best_svr_squat = make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)
best_svr_squat.fit(X_train.iloc[:20_000], y_squat_train.iloc[:20_000])

best_svr_deadlift = make_svr(
    C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon
)
best_svr_deadlift.fit(X_train.iloc[:20_000], y_deadlift_train.iloc[:20_000])

# evaluate on the full test set
print("Squat SVR:", evaluate(y_squat_test, best_svr_squat.predict(X_test)))
print("Deadlift SVR:", evaluate(y_deadlift_test, best_svr_deadlift.predict(X_test)))

### Final comparison: accuracy and training time

In [ ]:
# name, how to build the model, number of lifters used to train it (None = all)
model_builders = [
    ("linear", lambda: LinearRegression(), None),
    ("poly2", lambda: make_poly_pipeline(2), None),
    ("poly3", lambda: make_poly_pipeline(3), None),
    (
        "svr",
        lambda: make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon),
        20_000,
    ),
]

final_results = []
for target_name, (y_train, y_test) in targets.items():
    # gym rule: median ratio computed on the train set
    ratio = (y_train / X_train["Best3BenchKg"]).median()
    scores = evaluate(y_test, X_test["Best3BenchKg"] * ratio)
    final_results.append(
        {
            "model": "gym rule",
            "target": target_name,
            "n_train": len(X_train),
            "fit_seconds": 0,
            **scores,
        }
    )

    for model_name, build_model, n_train in model_builders:
        n = n_train or len(X_train)
        model = build_model()

        start = time.perf_counter()
        model.fit(X_train.iloc[:n], y_train.iloc[:n])
        fit_seconds = time.perf_counter() - start

        scores = evaluate(y_test, model.predict(X_test))  # full test set
        final_results.append(
            {
                "model": model_name,
                "target": target_name,
                "n_train": n,
                "fit_seconds": fit_seconds,
                **scores,
            }
        )

final_results_df = pd.DataFrame(final_results).round(3)
final_results_df

### Training time vs dataset size

In [ ]:
# training time for growing train sizes (squat only, time does not depend on the target)
train_sizes = [2_000, 5_000, 10_000, 20_000]

complexity_results = []
for n in train_sizes:
    X_small, y_small = X_train.iloc[:n], y_squat_train.iloc[:n]

    models = [
        ("linear", LinearRegression()),
        ("poly2", make_poly_pipeline(2)),
        ("svr", make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)),
    ]
    for model_name, model in models:
        start = time.perf_counter()
        model.fit(X_small, y_small)
        fit_seconds = time.perf_counter() - start

        mae = evaluate(y_squat_test, model.predict(X_test))["MAE"]  # full test set
        complexity_results.append(
            {"model": model_name, "n_train": n, "fit_seconds": fit_seconds, "MAE": mae}
        )

complexity_df = pd.DataFrame(complexity_results).round(3)
complexity_df

### Complexity Graph

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

# one line per model
for model_name, model_results in complexity_df.groupby("model"):
    ax.loglog(
        model_results["n_train"],
        model_results["fit_seconds"],
        marker="o",
        label=model_name,
    )

# reference line: time grows like n^2, starting from the first svr point
svr_results = complexity_df[complexity_df["model"] == "svr"]
n0, t0 = svr_results["n_train"].iloc[0], svr_results["fit_seconds"].iloc[0]
ax.loglog(
    train_sizes, [t0 * (n / n0) ** 2 for n in train_sizes], "k--", label="n² (theory)"
)

ax.set_xlabel("Number of training lifters")
ax.set_ylabel("Training time (s)")
ax.set_title("Training time vs dataset size")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

fig.savefig(
    ROOT / "reports" / "figures" / "training_time.png", dpi=200, bbox_inches="tight"
)
plt.show()

### Application on personal use

In [ ]:
# personal profile for the prediction (bench with a pause, like in a meet)
BENCH_KG = 110
BODYWEIGHT_KG = 71
AGE = 20
SEX = 1  # 1 = man, 0 = woman
TESTED = 1  # 1 = drug-tested category

me = pd.DataFrame(
    [
        {
            "Best3BenchKg": BENCH_KG,
            "BodyweightKg": BODYWEIGHT_KG,
            "Age": AGE,
            "Sex": SEX,
            "Tested": TESTED,
        }
    ]
)

# train the best simple model on the full train set
poly_squat = make_poly_pipeline(2).fit(X_train, y_squat_train)
poly_deadlift = make_poly_pipeline(2).fit(X_train, y_deadlift_train)

print("Predicted squat:", poly_squat.predict(me[FEATURES])[0].round(1), "kg")
print("Predicted deadlift:", poly_deadlift.predict(me[FEATURES])[0].round(1), "kg")

# for comparison: the gym rule
print("Gym rule squat:", round(BENCH_KG * ratio_squat, 1), "kg")
print("Gym rule deadlift:", round(BENCH_KG * ratio_deadlift, 1), "kg")

### Check against real lifters with a similar profile

In [ ]:
# median squat and deadlift of real lifters close to my profile
similar_men = dataset[
    (dataset["Sex"] == 1)
    & (dataset["Best3BenchKg"].between(105, 115))
    & (dataset["BodyweightKg"].between(66, 76))
]
print(len(similar_men), "similar lifters")
similar_men[["Best3SquatKg", "Best3DeadliftKg"]].median()